# Conflict 2: Political Labeling vs. Impartial Treatment
## Ethical Framework: Deontology (Kantian)

X/Twitter's open-sourced algorithm contains user labels including `author_is_democrat`, `author_is_republican`, `author_is_power_user`, and `author_is_elon`. These flags are used in the ranking pipeline.

The **deontological ethical constraint** (Kant's Categorical Imperative) demands: treat all users impartially. Two tweets of equal content quality should receive equal visibility regardless of the author's identity.

### The Tension
The algorithm uses identity-based signals (who you are) rather than purely content-based signals (what you said) to determine visibility. This creates asymmetric treatment that cannot be universalised.

## Setup

In [1]:
# !pip install z3-solver
from z3 import *

## Domain Model

We model 3 users each posting a tweet of **identical quality** (score = 70). The key difference is their labels:
- User 0: Elon Musk (platform owner, `author_is_elon = True`)
- User 1: Regular user, tagged as democrat
- User 2: Regular user, tagged as republican

The algorithm applies visibility boosts based on these labels.

In [2]:
s = Solver()

# Domain: 3 users, each posting one tweet
quality    = [Int(f'quality_{i}') for i in range(3)]
visibility = [Int(f'visibility_{i}') for i in range(3)]

# Labels from the actual X/Twitter codebase
is_elon       = [Bool(f'is_elon_{i}') for i in range(3)]
is_power_user = [Bool(f'is_power_user_{i}') for i in range(3)]
political_label = [Int(f'political_label_{i}') for i in range(3)]
# political_label: 0 = none, 1 = democrat, 2 = republican

# User 0: Elon Musk (platform owner)
s.add(is_elon[0] == True, is_power_user[0] == True)
s.add(political_label[0] == 0, quality[0] == 70)

# User 1: Regular user, democrat
s.add(is_elon[1] == False, is_power_user[1] == False)
s.add(political_label[1] == 1, quality[1] == 70)

# User 2: Regular user, republican
s.add(is_elon[2] == False, is_power_user[2] == False)
s.add(political_label[2] == 2, quality[2] == 70)

print("Scenario: 3 users, all with content quality = 70")
print("  User 0: Elon (is_elon=True, is_power_user=True)")
print("  User 1: Regular (democrat)")
print("  User 2: Regular (republican)")

Scenario: 3 users, all with content quality = 70
  User 0: Elon (is_elon=True, is_power_user=True)
  User 1: Regular (democrat)
  User 2: Regular (republican)


## Functional Goal: Apply Label-Based Visibility Boosts

This models the actual behaviour found in the codebase: special users receive amplified visibility on top of their base content quality.

- `author_is_elon` → +50 visibility boost
- `author_is_power_user` → +20 visibility boost
- Regular users → no boost

These are arbitrary numbers as the exact boost is not defined in the open source data

In [3]:
ELON_BOOST = 50
POWER_USER_BOOST = 20

for i in range(3):
    boost = If(is_elon[i], ELON_BOOST,
            If(is_power_user[i], POWER_USER_BOOST, 0))
    s.assert_and_track(
        visibility[i] == quality[i] + boost,
        f"FUNCTIONAL_apply_label_boost_user_{i}"
    )

print("Functional constraints added:")
print("  - Elon's visibility:   70 + 50 = 120")
print("  - Regular visibility:  70 + 0  = 70")

Functional constraints added:
  - Elon's visibility:   70 + 50 = 120
  - Regular visibility:  70 + 0  = 70


## Ethical Goal: Impartial Treatment (Deontological)

Kant's Categorical Imperative requires that rules be universalisable. Equal quality content must receive equal visibility, regardless of author identity.

Formalised from the module's fairness axiom:

$$\forall u_1 \forall u_2 \left( quality(u_1) = quality(u_2) \rightarrow visibility(u_1) = visibility(u_2) \right)$$

In [4]:
# Ethical Goal: equal quality → equal visibility
for i in range(3):
    for j in range(i + 1, 3):
        s.assert_and_track(
            Implies(
                quality[i] == quality[j],
                visibility[i] == visibility[j]
            ),
            f"ETHICAL_equal_quality_equal_visibility_{i}_{j}"
        )

print("Ethical constraints added:")
print("  - If quality[i] == quality[j], then visibility[i] == visibility[j]")
print("  - Applied to all user pairs: (0,1), (0,2), (1,2)")

Ethical constraints added:
  - If quality[i] == quality[j], then visibility[i] == visibility[j]
  - Applied to all user pairs: (0,1), (0,2), (1,2)


## Conflict Detection

In [5]:
print("CONFLICT 2: Political Labeling vs. Impartial Treatment")
print("Ethical Framework: Deontology (Kantian)")

result = s.check()

if result == sat:
    print("\nResult: SAT (No conflict)")
    m = s.model()
    for i in range(3):
        print(f"  User {i}: quality={m.evaluate(quality[i])}, "
              f"visibility={m.evaluate(visibility[i])}")

elif result == unsat:
    print("\nResult: UNSAT — Conflict detected!")
    core = s.unsat_core()
    print(f"\nUnsat Core ({len(core)} conflicting constraints):")
    for c in core:
        print(f"  - {c}")

    print("\nInterpretation:")
    print("  The functional goal (boost visibility for special users)")
    print("  conflicts with the deontological principle of impartiality.")
    print("  Elon's tweet gets visibility 120 while regular users get 70,")
    print("  despite all three tweets having identical quality (70).")
    print()
    print("  This violates Kant's Categorical Imperative:")
    print("  'Platform owners get special treatment' cannot be willed")
    print("  as a universal law for all social media platforms.")

CONFLICT 2: Political Labeling vs. Impartial Treatment
Ethical Framework: Deontology (Kantian)

Result: UNSAT — Conflict detected!

Unsat Core (3 conflicting constraints):
  - FUNCTIONAL_apply_label_boost_user_0
  - FUNCTIONAL_apply_label_boost_user_1
  - ETHICAL_equal_quality_equal_visibility_0_1

Interpretation:
  The functional goal (boost visibility for special users)
  conflicts with the deontological principle of impartiality.
  Elon's tweet gets visibility 120 while regular users get 70,
  despite all three tweets having identical quality (70).

  This violates Kant's Categorical Imperative:
  'Platform owners get special treatment' cannot be willed
  as a universal law for all social media platforms.


## Resolution Options

1. **Remove all identity-based boosts** — pure meritocracy based on content quality only
2. **Make boosts transparent and user-controllable** — let users decide if they want amplified content from power users
3. **Apply boosts only to content quality signals**, never to author identity signals